In [173]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import requests
import json 

RANDOM_STATE = 4150
np.random.seed(RANDOM_STATE)

In [174]:
# function to grab a game data
API_KEY = "C45399540985DC199A1C374C8EF89F6F"
GENRES_COUNT = 3
CATEGORIES_COUNT = 5

def GetGameInfo(appid, playtime, is_installed):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=us"
    data = requests.get(url).json()
    game = data[str(appid)]["data"]
    result = []

    #result.append(game["name"])
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([game["name"]])
    result.append(tfidf_matrix[0, 0])
    #result.append(game["publishers"][0])
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([game["publishers"][0]])
    result.append(tfidf_matrix[0, 0])
    for i in range(0, GENRES_COUNT):
        if i >= len(game["genres"]):
            result.append(0)
        else:
            result.append(int(game["genres"][i]['id']))
    for platform in game["platforms"]:
        result.append(game["platforms"][platform])
    for i in range(0, CATEGORIES_COUNT):
        if i >= len(game["categories"]):
            result.append(0)
        else:
            result.append(int(game["categories"][i]['id']))
    result.append(int(game["release_date"]["date"][-4:]))
    price_overview = game.get("price_overview")
    if price_overview:
        result.append(game["price_overview"]["final"])
    else:
        result.append(0)
    result.append(playtime)
    result.append(is_installed)

    return result

In [175]:
# Test to get player data
print(GetGameInfo(1306630, 11.8, False))
print("-----")
print(GetGameInfo(537110, 6.6, False))
print("-----")
print(GetGameInfo(2230650, 93.6, True))
print("-----")
print(GetGameInfo(3887700, 152, False))
print("-----")
print(GetGameInfo(648100, 35.4, False))
print("-----")
print(GetGameInfo(730, 2428, True))
print("-----")
print(GetGameInfo(1808500, 142.1, True))
print("-----")
print(GetGameInfo(1174180, 24.3, False))
print("-----")
print(GetGameInfo(2807960, 13.1, True))
print("-----")
print(GetGameInfo(3809440, 2.1, True))
print("-----")

[np.float64(0.7071067811865475), np.float64(0.7071067811865475), 1, 25, 23, True, True, True, 2, 22, 28, 55, 56, 2021, 799, 11.8, False]
-----
[np.float64(0.5773502691896258), np.float64(0.5773502691896258), 25, 23, 3, True, False, False, 2, 29, 18, 62, 0, 2016, 599, 6.6, False]
-----
[np.float64(1.0), np.float64(1.0), 1, 25, 4, True, False, False, 2, 22, 28, 55, 56, 2023, 1499, 93.6, True]
-----
[np.float64(0.7071067811865475), np.float64(0.5773502691896258), 3, 2, 37, True, False, False, 2, 22, 35, 62, 0, 2025, 0, 152, False]
-----
[np.float64(0.7071067811865475), np.float64(1.0), 25, 23, 0, True, False, False, 2, 18, 62, 0, 0, 2017, 1199, 35.4, False]
-----
[np.float64(0.7071067811865475), np.float64(1.0), 1, 37, 0, True, False, True, 1, 27, 29, 51, 30, 2012, 0, 2428, True]
-----
[np.float64(0.7071067811865475), np.float64(0.7071067811865475), 1, 0, 0, True, False, False, 1, 49, 36, 9, 38, 2025, 1999, 142.1, True]
-----
[np.float64(0.5773502691896258), np.float64(0.7071067811865475)

In [176]:
# Activation Mapping
RESULT_MAPPING = ['Lost Ruins', 'Angels of Death', 'TEVI', "Girls' Frontline", 'Raging Loop', 'Momodora: Moonlit Farewell',
                'Counter-Strike 2', 'ARC Raiders', 'Red Dead Redemption 2', 'Battlefield™ 6', 'Cheese Rolling', "Tom Clancy's Rainbow Six Siege",]

print(RESULT_MAPPING[5])
print(RESULT_MAPPING[11])

Momodora: Moonlit Farewell
Tom Clancy's Rainbow Six Siege


In [177]:
# Make training data
# Idea: 5 provided -> 1 expect result
# Game ID, Playtime(Hours), Is_Installed
PLAYER1_INPUT = [[1306630, 11.8, False], [537110, 6.6, False], [2230650, 93.6, True], [3887700, 152, False], [648100, 35.4, False]]
PLAYER1_OUTPUT = 5
PLAYER2_INPUT = [[730, 2428, True], [1808500, 142.1, True], [1174180, 24.3, False], [2807960, 13.1, True], [3809440, 2.1, True]]
PLAYER2_OUTPUT = 11

train_input = []
train_output = []

train_input.append([])
for ele in PLAYER1_INPUT:
    gameInfo = GetGameInfo(ele[0], ele[1], ele[2])
    for gamePiece in gameInfo:
        train_input[0].append(gamePiece)
player_output = [0.0] * len(RESULT_MAPPING)
player_output[PLAYER1_OUTPUT] = 1.0
train_output.append(player_output)

train_input.append([])
for ele in PLAYER2_INPUT:
    gameInfo = GetGameInfo(ele[0], ele[1], ele[2])
    for gamePiece in gameInfo:
        train_input[1].append(gamePiece)
player_output = [0.0] * len(RESULT_MAPPING)
player_output[PLAYER2_OUTPUT] = 1.0
train_output.append(player_output)

print(train_input)
print(train_output)

[[np.float64(0.7071067811865475), np.float64(0.7071067811865475), 1, 25, 23, True, True, True, 2, 22, 28, 55, 56, 2021, 799, 11.8, False, np.float64(0.5773502691896258), np.float64(0.5773502691896258), 25, 23, 3, True, False, False, 2, 29, 18, 62, 0, 2016, 599, 6.6, False, np.float64(1.0), np.float64(1.0), 1, 25, 4, True, False, False, 2, 22, 28, 55, 56, 2023, 1499, 93.6, True, np.float64(0.7071067811865475), np.float64(0.5773502691896258), 3, 2, 37, True, False, False, 2, 22, 35, 62, 0, 2025, 0, 152, False, np.float64(0.7071067811865475), np.float64(1.0), 25, 23, 0, True, False, False, 2, 18, 62, 0, 0, 2017, 1199, 35.4, False], [np.float64(0.7071067811865475), np.float64(1.0), 1, 37, 0, True, False, True, 1, 27, 29, 51, 30, 2012, 0, 2428, True, np.float64(0.7071067811865475), np.float64(0.7071067811865475), 1, 0, 0, True, False, False, 1, 49, 36, 9, 38, 2025, 1999, 142.1, True, np.float64(0.5773502691896258), np.float64(0.7071067811865475), 1, 25, 0, True, False, False, 2, 1, 49, 36, 

In [178]:
# Make testing data
TEST_INPUT = [[620, 300, True], [2570630, 200, True], [288160, 100, False], [425580, 100, True], [570, 10, True]]
TEST_OUTPUT = 1
testing_input = []
testing_output = []
testing_input.append([])
for ele in TEST_INPUT:
    gameInfo = GetGameInfo(ele[0], ele[1], ele[2])
    for gamePiece in gameInfo:
        testing_input[0].append(gamePiece)
player_output = [0] * len(RESULT_MAPPING)
player_output[TEST_OUTPUT] = 1
testing_output.append(player_output)

In [179]:
def train_model(hidden_layer_sizes=(4,), activation="relu", alpha=0.0001, max_iter=1000):
    """Train a simple neural network classifier and return model + metrics."""
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        alpha=alpha,
        max_iter=max_iter,
        random_state=RANDOM_STATE
    )
    model.fit(train_input, train_output)

    train_pred = model.predict_proba(train_input)
    res_output = []
    for one_array in train_pred:
        this_res = []
        for i, res in enumerate(one_array):
            if res > 0.5:
                this_res.append((RESULT_MAPPING[i], res))
        res_output.append(this_res)
    print(res_output)

    test_pred = model.predict_proba(testing_input)
    res_output = []
    for one_array in test_pred:
        this_res = []
        for i, res in enumerate(one_array):
            if res > 0.5:
                this_res.append((RESULT_MAPPING[i], res))
        this_res.sort(key=lambda x: x[1], reverse=True)
        res_output.append(this_res)
    print(res_output)

    # results = {
    #     "hidden_layer_sizes": hidden_layer_sizes,
    #     "activation": activation,
    #     "alpha": alpha,
    #     "iterations": model.n_iter_,
    #     "train_accuracy": accuracy_score(train_output, train_pred),
    #     "test_accuracy": accuracy_score(testing_output, test_pred),
    #     "train_loss_final": model.loss_,
    # }
    return model#, results

In [180]:
# Run model
#baseline_model, baseline_results = train_model(hidden_layer_sizes=(16,16), activation="relu", alpha=0.1)
baseline_model = train_model(hidden_layer_sizes=(16,16), activation="relu", alpha=0.1)


[[('Momodora: Moonlit Farewell', np.float64(0.9999999284660035))], [("Tom Clancy's Rainbow Six Siege", np.float64(1.0))]]
[[('Angels of Death', np.float64(0.9999999146744516)), ('Red Dead Redemption 2', np.float64(0.9999938318260656)), ('Counter-Strike 2', np.float64(0.9999899712081942)), ('Momodora: Moonlit Farewell', np.float64(0.9932756992265183)), ("Tom Clancy's Rainbow Six Siege", np.float64(0.9856916551013792))]]
